# Sentiment Analysis Using Bag of Words and XGBoost

## Project Overview

Natural Language Processing (NLP) enables machines to understand, analyze, and extract meaningful information from human language. One of the most fundamental NLP tasks is **Sentiment Analysis**, which aims to determine whether a piece of text expresses a positive or negative opinion.

In this project, I build a complete sentiment analysis pipeline using classical NLP techniques and Machine Learning algorithms. The workflow covers the entire process from raw text preprocessing to feature extraction, model training, evaluation, and experiment documentation.

The primary objective of this project is not only to achieve high predictive performance but also to systematically investigate how different preprocessing techniques, feature engineering strategies, and machine learning models affect sentiment classification performance.

---

## Dataset

Dataset Link: **[IMBD Movie Reviews](https://www.kaggle.com/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis)**

The dataset consists of movie reviews labeled with their corresponding sentiment:

* Positive
* Negative

The reviews contain real-world natural language, making them suitable for evaluating text preprocessing techniques and machine learning models.

---

## Project Pipeline

The following stages are implemented throughout this project:

### 1. Text Preprocessing

* Contraction expansion
* Lowercasing
* Text cleaning using regular expressions
* Stopword removal with preserved negations
* Lemmatization
* Corpus construction

### 2. Feature Extraction

* Bag of Words (CountVectorizer)
* N-gram generation
* Vocabulary analysis
* Feature selection using `max_features`
* Rare-word filtering using `min_df`

### 3. Model Training

Different machine learning algorithms are evaluated and compared, including:

* Gaussian Naive Bayes
* Multinomial Naive Bayes
* Bernoulli Naive Bayes
* Logistic Regression
* Support Vector Machines (SVM)
* Decision Trees
* Random Forests

### 4. Model Evaluation

Performance is assessed using multiple metrics:

* Accuracy
* Precision
* Recall
* F1-Score
* ROC-AUC Score
* Confusion Matrix
* Cross-Validation Mean Accuracy
* Cross-Validation Standard Deviation

---

## Experimental Approach

Rather than training a single model, this notebook follows an experimentation-driven methodology.

For each model, multiple configurations are tested, including different values for:

* `max_features`
* `min_df`
* N-gram ranges
* Preprocessing strategies

The goal is to identify the most effective configuration while understanding the trade-offs between model complexity, computational cost, and predictive performance.

---

## Key Learning Objectives

Through this project, I aim to:

* Develop a deeper understanding of NLP preprocessing techniques.
* Compare the behavior of different machine learning algorithms on text data.
* Analyze how feature engineering impacts classification performance.
* Build reproducible NLP pipelines suitable for real-world applications.
* Establish strong baselines before moving toward advanced embedding and transformer-based approaches.

---

**Author:** Hazem Mohamed

**Role:** AI Engineer | Machine Learning Engineer | NLP Engineer

**Repository:** [NLP Experimentation Lab](https://github.com/Hazem1695/NLP-Experimentation-Lab)


# **Importing the Libraries**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# **Data preprocessing**

## Data Cleaning Check Template
This template is designed to quickly assess the quality of any dataset before building machine learning models or performing analysis.

It provides a structured overview of the dataset by checking for common data issues such as:

- Missing values

- Duplicate rows

- Incorrect data types

- Outliers

- Distribution of numerical features

- Categorical feature consistency

**What This Template Does**

- Displays basic dataset information (shape, data types)

- Identifies missing values and duplicates

- Summarizes numerical and categorical features

- Detects potential outliers using the IQR method

- Highlights columns with low unique values for quick inspection

How to Use

1. Load your dataset using Pandas  

2. Call the function:

In [2]:
def data_quality_report(df):

    print("DATA QUALITY REPORT")
    
    # Print a separator line for better readability
    
    print("=" * 50)
    print("BASIC INFO")
    print("=" * 50)
    
    # Show general information about the dataset (columns, data types, non-null values)
    print(df.info())
    
    # Show number of rows and columns
    print("\n" + "=" * 50)
    print("SHAPE OF DATA")
    print("=" * 50)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for missing (null) values in each column
    print("\n" + "=" * 50)
    print("MISSING VALUES")
    print("=" * 50)
    missing = df.isnull().sum()
    
    # Display only columns that have missing values
    print(missing[missing > 0])
    
    # Check for duplicate rows
    print("\n" + "=" * 50)
    print("DUPLICATES")
    print("=" * 50)
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    # Display data types of each column
    print("\n" + "=" * 50)
    print("DATA TYPES")
    print("=" * 50)
    print(df.dtypes)
    
    # Summary statistics for numerical columns (mean, std, min, max, etc.)
    print("\n" + "=" * 50)
    print("NUMERICAL SUMMARY")
    print("=" * 50)
    print(df.describe())
    
    # Summary for categorical (object) columns
    print("\n" + "=" * 50)
    print("CATEGORICAL SUMMARY")
    print("=" * 50)
    print(df.describe(include=['object']))
    
    # Show unique values for columns with low number of distinct values
    # Useful for detecting categories, errors, or inconsistencies
    print("\n" + "=" * 50)
    print("UNIQUE VALUES (LOW CARDINALITY)")
    print("=" * 50)
    for col in df.columns:
        if df[col].nunique() < 10:  # Only show columns with few unique values
            print(f"{col}: {df[col].unique()}")
            
    # correlation
    print("\n" + "=" * 50)
    print("CORRELATION MATRIX")
    print("=" * 50)
    print(df.corr(numeric_only=True))
    
    # Detect outliers using the IQR (Interquartile Range) method
    print("\n" + "=" * 50)
    print("OUTLIERS CHECK (IQR METHOD)")
    print("=" * 50)
    
    # Loop through only numerical columns
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range
        
        # Count rows that fall outside the normal range
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers")

## **Load dataset**
Apply Data Cleaning Check Template

In [ ]:
dataset = pd.read_csv('MovieReviewTrainingDatabase.csv')
data_quality_report(dataset)

DATA QUALITY REPORT
BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  25000 non-null  object
 1   review     25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB
None

SHAPE OF DATA
Rows: 25000, Columns: 2

MISSING VALUES
Series([], dtype: int64)

DUPLICATES
Duplicate rows: 96

DATA TYPES
sentiment    object
review       object
dtype: object

NUMERICAL SUMMARY
       sentiment                                             review
count      25000                                              25000
unique         2                                              24904
top     Positive  You do realize that you've been watching the E...
freq       12500                                                  3

CATEGORICAL SUMMARY
       sentiment                                             review
count      25000                 

## Duplicate Data Detection

In [4]:
duplicates = dataset[dataset.duplicated(subset=['review'], keep=False)]
duplicates.sort_values('review')

,sentiment,review
21186,Negative,"Back in his youth, the old man had wanted to..."
21877,Negative,"Back in his youth, the old man had wanted to..."
14734,Negative,'Dead Letter Office' is a low-budget film abou...
5519,Negative,'Dead Letter Office' is a low-budget film abou...
7011,Positive,".......Playing Kaddiddlehopper, Col San Fernan..."
...,...,...
2685,Negative,"in this movie, joe pesci slams dunks a basketb..."
22244,Positive,it's amazing that so many people that i know h...
14767,Positive,it's amazing that so many people that i know h...
12462,Negative,this movie begins with an ordinary funeral... ...


## Quantifying Duplicate Review Frequencies

In [5]:
review_counts = dataset['review'].value_counts()
print("Reviews appearing more than once:")
print((review_counts > 1).sum())
print("\nMaximum repetitions:")
print(review_counts.max())

Reviews appearing more than once:
92

Maximum repetitions:
3


## Removing Duplicate Reviews & Resetting Index
> **Note:** This cell drops the repeated rows we identified in the previous steps and cleanly resets the row indices for model training

In [6]:
print("Before:", len(dataset))
dataset = dataset.drop_duplicates()
print("After:", len(dataset))
dataset = dataset.reset_index(drop=True)

Before: 25000
After: 24904


## Library Installation
> **Note:** The `contractions` library is required to automatically expand shortcuts like *don't* to *do not* and *I'm* to *I am* during preprocessing.

In [7]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 2.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.8 MB/s eta 0:00:00


## **Cleaning the texts**

In [8]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
import re
import contractions
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

base_stopwords = set(stopwords.words('english'))
negation_words = {'not', 'no', 'never'}
all_stopwords = base_stopwords - negation_words

wnl = WordNetLemmatizer()

corpus = []

for i in range(0, len(dataset)):
    review = dataset['review'][i]
    # Fix contractions
    review = contractions.fix(review)
    # Lowercase
    review = review.lower()
    
    review = re.sub(r'[^a-zA-Z\s]', ' ', review)
    # Split
    words = review.split()
    # Chained Lemmatization (Handles both Verbs 'v' and Nouns 'n')
    review = [wnl.lemmatize(wnl.lemmatize(word, pos='v'), pos='n') for word in words if word not in all_stopwords]
    review = ' '.join(review) 
    corpus.append(review)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


## Preprocessing Verification
> **Note:** Pulling the first two rows directly as a memory array to confirm that our lowercasing, stopword stripping, and lemmatization pipeline worked correctly before feeding it into the vectorizer.

In [9]:
# Pull the data directly as a fast memory array
raw_samples = dataset['review'].head(2).values

for i in range(2):
    print(f"=== REVIEW #{i+1} ===")
    print(f"RAW:     {raw_samples[i]}\n") 
    print(f"CLEANED: {corpus[i]}")
    print("-" * 50)

=== REVIEW #1 ===
RAW:     With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.  Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.  The actual feature film bit when it final

# **Encoding Categorical data Using Label Encoding**

In [10]:
from sklearn.preprocessing import LabelEncoder
y = dataset.iloc[:, 0].values
le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
print(y)

[1 1 0 ... 0 0 1]


# Class Balance Check
> **Note:** Using NumPy to verify if our dataset is perfectly balanced between positive and negative reviews before splitting it into training and testing sets.

In [12]:
# This returns the unique classes and how many times they appear
classes, counts = np.unique(y, return_counts=True)
for c, count in zip(classes, counts):
    print(f"Class {c} contains {count}")

Class 0 contains 12432
Class 1 contains 12472


# **Splitting the dataset into the Training set and Test set**

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(corpus, y, test_size = 0.20, random_state = 0)

# **Creating the Bag of Word model**

In [14]:
from sklearn.feature_extraction.text import CountVectorizer
# Add ngram_range=(1, 2) so it automatically catches phrases like "not good" or "no clue"
cv = CountVectorizer(max_features=10000, ngram_range=(1,2)) # first try without (max_features)
X_train = cv.fit_transform(X_train)
X_test = cv.transform(X_test)

In [15]:
print(X_train.shape)        # (n_samples, n_features)
print(X_train.shape[0])     # number of training samples
print(X_train.shape[1])     # number of features (vocabulary size)

(19923, 10000)
19923
10000


In [16]:
print(X_test.shape)        # (n_samples, n_features)
print(X_test.shape[0])     # number of training samples
print(X_test.shape[1])     # number of features (vocabulary size)

(4981, 10000)
4981
10000


# Tuning Hyperparameters with HalvingRandomSearchCV

In [17]:
from xgboost import XGBClassifier
from sklearn.experimental import enable_halving_search_cv  # Required
from sklearn.model_selection import HalvingRandomSearchCV
from scipy.stats import uniform, randint

# Initialize XGBoost
xgb = XGBClassifier(
    random_state=0,
    eval_metric='logloss'
)

# Define parameter grid
param_grid = {
    "learning_rate": uniform(0.01, 0.19),      # samples continuously from 0.01 to 0.20
    "max_depth": randint(3, 8),                 # samples integers from 3 to 7
    "min_child_weight": randint(1, 6),          # samples integers from 1 to 5
    "subsample": uniform(0.7, 0.3),             # samples from 0.7 to 1.0
    "colsample_bytree": uniform(0.7, 0.3)       # samples from 0.7 to 1.0
}

# Halving Random Search
halving_search = HalvingRandomSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    factor=3,                # Controls how aggressively candidates are reduced
    resource='n_estimators',  # Progressively increase number of boosting rounds
    min_resources=20,         # Starting number of trees for early elimination rounds
    max_resources=500,        # Final round trains up to 500 trees
    scoring='accuracy',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

# Fit on training data
halving_search.fit(X_train, y_train)

# Best parameters and score
print("Best Parameters:", halving_search.best_params_)
print("Best CV Score:", halving_search.best_score_)

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 20
max_resources_: 500
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 25
n_resources: 20
Fitting 3 folds for each of 25 candidates, totalling 75 fits
----------
iter: 1
n_candidates: 9
n_resources: 60
Fitting 3 folds for each of 9 candidates, totalling 27 fits
----------
iter: 2
n_candidates: 3
n_resources: 180
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best Parameters: {'colsample_bytree': np.float64(0.8525712073494107), 'learning_rate': np.float64(0.18243763004595767), 'max_depth': 7, 'min_child_weight': 3, 'subsample': np.float64(0.7609183674204307), 'n_estimators': 180}
Best CV Score: 0.8609145209054861
[CV] END colsample_bytree=0.8123620356542087, learning_rate=0.19063571821788408, max_depth=5, min_child_weight=5, n_estimators=20, subsample=0.879055047383946; total time=   1.4s
[CV] END colsample_bytree=0.8337498258560773, learning_rate=0.028995234005420548, 

# **Training the XGBoost model on the Training set**

In [17]:
from xgboost import XGBClassifier
classifier = XGBClassifier(booster='gbtree', eval_metric='logloss', subsample=0.7609183674204307, n_estimators=1000, min_child_weight=3, max_depth=7, learning_rate=0.18243763004595767, colsample_bytree=0.8525712073494107, random_state = 0)
classifier.fit(X_train, y_train)

XGBClassifier(base_score=None, booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8525712073494107, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.18243763004595767,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None,
              num_parallel_tree=None, ...)

# **Predicting the Test set results**

In [18]:
y_pred = classifier.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[1 1]
 [1 1]
 [0 0]
 ...
 [0 0]
 [0 0]
 [0 0]]


# **Evaluating the Model Performance**

In [19]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_pred))


accuracies = cross_val_score(estimator=classifier, X=X_train, y=y_train, cv=3)

print("\nMean Accuracy:")
print(accuracies.mean())

print("\nStandard Deviation:")
print(accuracies.std())

Confusion Matrix:
[[2159  361]
 [ 264 2197]]

Accuracy Score:
0.8745231881148364

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.86      0.87      2520
           1       0.86      0.89      0.88      2461

    accuracy                           0.87      4981
   macro avg       0.87      0.87      0.87      4981
weighted avg       0.88      0.87      0.87      4981


ROC-AUC Score:
0.8747362828376644

Mean Accuracy:
0.8679917683079857

Standard Deviation:
0.0026292829099450396


# XGBoost Performance Analysis for Text Classification

# 1. Objective

The objective of this experiment was to evaluate the effectiveness of **XGBoost** for binary text classification using features generated by **CountVectorizer**, and to compare its performance against the previously evaluated Decision Tree and Random Forest classifiers.

As a gradient-boosted ensemble, XGBoost builds trees sequentially, with each new tree correcting the errors of the previous ones — a fundamentally different ensembling strategy than Random Forest's parallel, independent trees. This study investigates booster type, vocabulary size, and hyperparameter optimization, including a case where tuning initially hurt performance before a corrected search found the best result of the series.

The experiment aims to answer the following research questions:

* How does XGBoost compare to Decision Tree and Random Forest on the same features?
* Does the `gbtree` booster outperform the linear `gblinear` booster for this task?
* How does vocabulary size affect performance?
* Can `HalvingRandomSearchCV` reliably improve on untuned defaults, and what happens when it doesn't?
* Do the final evaluated models match what the hyperparameter search actually selected?

---

# 2. Experimental Setup

## Dataset

* Final test set: **4,981 documents** (2,520 negative / 2,461 positive).

## Feature Extraction

Documents were transformed into numerical vectors using **CountVectorizer**.

| Configuration | Parameters                            |
| -------------- | -------------------------------------- |
| C1             | max_features=5000, ngram_range=(1,2)   |
| C2             | max_features=10000, ngram_range=(1,2)  |
| C3             | max_features=15000, ngram_range=(1,2)  |

---

# 3. Hyperparameter Optimization Strategy

Two separate `HalvingRandomSearchCV` rounds were run over the course of this experiment, using different parameter grids and different halving resources.

**Round 1** used a coarse grid and `resource='n_samples'` (successive halving allocates increasingly larger *training-sample subsets* to surviving candidates):

```python
param_grid = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}
```

**Round 2** used continuous distributions and switched to `resource='n_estimators'` (successive halving allocates increasingly more *boosting rounds* to surviving candidates), which is generally a better fit for gradient boosting since the number of trees is the natural "more resources = more training" axis:

```python
param_grid = {
    "learning_rate": uniform(0.01, 0.19),
    "max_depth": randint(3, 8),
    "min_child_weight": randint(1, 6),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3)
}
```

As detailed in Section 6, Round 1 produced models that underperformed the untuned defaults, while Round 2 produced the best result of the entire experiment.

---

# 4. Hyperparameters Explored and Selected

### Round 1 — Best Parameters (5K and 10K features, identical result)

```python
{'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 1,
 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
```

Both feature sizes converged on the same configuration, with `learning_rate=0.01` at the floor of the search space.

### Round 2 — Best Parameters (10K features only)

```python
{'colsample_bytree': 0.8526, 'learning_rate': 0.1824, 'max_depth': 7,
 'min_child_weight': 3, 'subsample': 0.7609, 'n_estimators': 180}
```

> ⚠️ **Discrepancy:** the final evaluated Round 2 model used `n_estimators=1000`, not the `n_estimators=180` the search actually selected — every other parameter matches. The 1000-tree version does score highest overall (Section 6).

No tuned run was performed for the 15K-feature configuration — only the untuned default `gbtree` model was evaluated at that vocabulary size.

---

# 5. Booster Type Comparison: `gbtree` vs. `gblinear`

At 5,000 features, both boosting strategies were evaluated untuned:

| Booster    | Accuracy | ROC-AUC |
| ---------- | -------- | ------- |
| **gbtree** | **85.55%** | **0.8557** |
| gblinear   | 82.63%   | 0.8265  |

`gbtree` outperformed `gblinear` by nearly 3 points. This is expected: `gblinear` fits a linear model via boosted linear updates, which doesn't capture feature interactions the way tree-based splits can — for sparse BoW text data with thousands of dimensions, tree-based boosting has more capacity to exploit combinations of word/n-gram features that a purely linear booster cannot represent. All subsequent tuning in this experiment used `gbtree` only.

---

# 6. Experimental Results

| Configuration                              | Accuracy   | ROC-AUC    | CV Mean | CV Std |
| -------------------------------------------- | ---------- | ---------- | ------- | ------ |
| 5K, gbtree (default)                         | 85.55%     | 0.8557     | 85.09%  | 0.0028 |
| 5K, gblinear (default)                       | 82.63%     | 0.8265     | 82.10%  | 0.0029 |
| 5K, Round 1 tuned                            | 80.59%     | 0.8066     | 80.38%  | 0.0050 |
| 10K, gbtree (default)                        | 86.39%     | 0.8641     | 85.40%  | 0.0031 |
| 10K, Round 1 tuned                           | 80.63%     | 0.8070     | 80.41%  | 0.0050 |
| 10K, Round 1 params + manual lr=0.08         | 87.03%     | 0.8706     | 86.44%  | 0.0015 |
| 10K, Round 2 tuned (n_estimators overridden) | **87.45%** | **0.8747** | 86.80%  | 0.0026 |
| 15K, gbtree (default)                        | 86.01%     | 0.8603     | 85.17%  | 0.0020 |

---

# 7. Performance Analysis

## When Tuning Hurts: Round 1's Underperformance

Round 1's tuned models scored roughly **5 points lower** than the untuned `gbtree` defaults at both 5K (80.59% vs. 85.55%) and 10K (80.63% vs. 86.39%). The common thread is `learning_rate=0.01` — the lowest value in the search grid. With a fixed `n_estimators=500`, a learning rate that low likely left the ensemble undertrained (each tree contributing too little to the final prediction within the available boosting rounds). This is reinforced directly by the next result:

## Confirming the Cause: Manual Learning-Rate Adjustment

Keeping every other Round 1 parameter fixed and manually raising `learning_rate` from 0.01 to 0.08 lifted 10K accuracy from 80.63% to **87.03%** — a jump of over 6 points from a single hyperparameter change. This strongly suggests the low learning rate, not the other parameters, was responsible for Round 1's underperformance, and that `resource='n_samples'` combined with a grid bottoming out at 0.01 was a poor search design for this problem.

## Round 2's Improved Search Design

Round 2 switched to `resource='n_estimators'` (a more natural fit for gradient boosting) and a wider, continuous learning-rate range starting at 0.01 but able to reach up to 0.20. It selected `learning_rate≈0.18`, `max_depth=7` — a deeper, faster-learning configuration than Round 1 ever considered. The resulting model (with `n_estimators` overridden to 1000) achieved the best result of the experiment at 87.45%, about 1 point above the untuned 10K default.

## Effect of Vocabulary Size (Untuned Models)

| Features | Accuracy (default gbtree) |
| -------- | -------------------------- |
| 5K       | 85.55%                     |
| **10K**  | **86.39%**                 |
| 15K      | 86.01%                     |

The same rise-then-slight-decline pattern seen in the Decision Tree and Random Forest experiments holds here too: 10K is the peak, with 15K giving marginally less than 10K.

---

# 8. Precision and Recall Analysis

### Best Model — 10K, Round 2 Tuned

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0     | 0.89      | 0.86   | 0.87     |
| 1     | 0.86      | 0.89   | 0.88     |

### 10K, Manual lr=0.08

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0     | 0.89      | 0.84   | 0.87     |
| 1     | 0.85      | 0.90   | 0.87     |

### 10K, Default gbtree

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0     | 0.88      | 0.84   | 0.86     |
| 1     | 0.85      | 0.89   | 0.87     |

### 10K, Round 1 Tuned (underperforming)

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0     | 0.85      | 0.75   | 0.80     |
| 1     | 0.77      | 0.87   | 0.82     |

The Round 1 underfit model shows a much wider precision/recall gap between classes (10 points of recall difference vs. 2-6 points in the properly tuned models) — a useful diagnostic signature of an undertrained boosting model, worth remembering for future tuning runs.

---

# 9. ROC-AUC Analysis

| Configuration                      | ROC-AUC    |
| ------------------------------------ | ---------- |
| 5K, gbtree (default)                 | 0.8557     |
| 5K, gblinear (default)               | 0.8265     |
| 5K, Round 1 tuned                    | 0.8066     |
| 10K, gbtree (default)                | 0.8641     |
| 10K, Round 1 tuned                   | 0.8070     |
| 10K, manual lr=0.08                  | 0.8706     |
| **10K, Round 2 tuned**               | **0.8747** |
| 15K, gbtree (default)                | 0.8603     |

ROC-AUC tracks accuracy closely throughout, including the Round 1 dip.

---

# 10. Cross-Validation Analysis

| Configuration                | CV Mean Accuracy | CV Std Dev |
| ------------------------------ | ----------------- | ---------- |
| 5K, gbtree (default)            | 85.09%             | 0.0028     |
| 5K, gblinear (default)          | 82.10%             | 0.0029     |
| 5K, Round 1 tuned               | 80.38%             | 0.0050     |
| 10K, gbtree (default)           | 85.40%             | 0.0031     |
| 10K, Round 1 tuned              | 80.41%             | 0.0050     |
| 10K, manual lr=0.08             | 86.44%             | **0.0015** |
| **10K, Round 2 tuned**          | **86.80%**         | 0.0026     |
| 15K, gbtree (default)           | 85.17%             | 0.0020     |

Interestingly, the manual lr=0.08 configuration has the tightest CV spread of any tuned run (0.0015), even tighter than the best-accuracy Round 2 model (0.0026). Combined with its already strong accuracy (87.03%, only 0.42 points behind the best model), this makes it a reasonable "runner-up" candidate if stability is a priority.

---

# 11. Best Model Configuration

The highest-scoring evaluated model was:

```python
CountVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

XGBClassifier(
    booster='gbtree',
    eval_metric='logloss',
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.18243763004595767,
    min_child_weight=3,
    subsample=0.7609183674204307,
    colsample_bytree=0.8525712073494107,
    random_state=0
)
```

Performance:

* Accuracy = **87.45%**
* Precision (Class 0 / 1) = **0.89 / 0.86**
* Recall (Class 0 / 1) = **0.86 / 0.89**
* ROC-AUC = **0.8747**
* Cross-Validation Accuracy = **86.80%**
* Cross-Validation Standard Deviation = **0.0026**

---

# 12. Comparison with Previously Evaluated Models (CountVectorizer)

| Model                     | Best Accuracy  | ROC-AUC    |
| ------------------------- | -------------- | ---------- |
| **XGBoost**               | **87.45%**     | **0.8747** |
| Random Forest             | *86.13%*         | *0.8614*     |
| Logistic Regression       | ***88.78%***      | ***0.8880***    |
| LinearSVC                 | *87.51%*      | *0.8753*  |
| Naive Bayes (BernoulliNB) | *87.61%*       | *0.8763*  |
| K-Nearest Neighbors       | *76.79%*      | *0.7671*  |
| Decision Tree             | *74.34%*         | *0.7441*     |


XGBoost's best result edges out Random Forest by about 1.3 points, consistent with gradient boosting's typical edge over bagging-based ensembles when properly tuned.

---

# 13. Discussion

This experiment tells two intertwined stories: XGBoost's ceiling on this dataset, and a cautionary tale about hyperparameter search design.

On the modeling side, `gbtree` clearly outperforms `gblinear`, confirming that boosting benefits from tree-based nonlinearity on sparse BoW features just as Random Forest did. XGBoost's best tuned result (87.45%) is the strongest of any classifier evaluated on CountVectorizer features so far, edging out Random Forest.

On the tuning side, Round 1 is a useful negative result: a search grid that happened to bottom out at an overly conservative `learning_rate=0.01`, combined with a `resource='n_samples'` halving strategy not particularly well-suited to gradient boosting, produced models that underperformed simple defaults by roughly 5 points. This wasn't a subtle effect — it's the kind of result that could easily be mistaken for "tuning doesn't help this model" if the untuned baseline weren't also on hand for comparison, which is a good argument for always keeping an untuned baseline in the results table. Round 2's redesigned search (continuous ranges, `resource='n_estimators'`) recovered and then improved on the untuned baseline.

---

# 14. Final Conclusion

This experiment evaluated XGBoost performance on binary text classification using CountVectorizer features, testing both booster types, three vocabulary sizes, and two rounds of `HalvingRandomSearchCV` tuning.

The best-performing model used **CountVectorizer with 10,000 features and bigrams**, paired with a **gbtree XGBClassifier** (max_depth=7, learning_rate≈0.18, 1000 estimators), achieving **87.45% accuracy** and **0.8747 ROC-AUC** — the strongest result of any classifier tested on CountVectorizer features to date, ahead of Random Forest (86.13%) and well ahead of Decision Tree (74.34%).